# 00 · Your first typed decision

**You will:** verify your kernel, make a Jev call, and identify the evidence, question and answer.

Run the README setup first. Every notebook is independent: restart its kernel and run from the top. Live requests use your account and incur charges. Only synthetic course data is sent.

A support ticket arrives: **“I was charged twice. Please refund the duplicate.”** Before running the cell, decide what you need: a refund-request flag, a written reply, or the refund transaction itself? These are different tasks.

In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

In [ ]:
# Show installed package versions and the Python interpreter used by this kernel.
from importlib.metadata import version
for package in ['typesafe-sdk', 'openai', 'pydantic', 'ipykernel']:
    print(package, version(package))
print('Kernel executable:', sys.executable)

## Evidence + a focused question

`state` is the evidence. `questions` defines the judgments. Question IDs are for your code; put the actual question in `instructions`. The model's response is not a command to refund money.

This introductory call is live-only. If the key is missing, the cell explains what to configure. Later notebooks add explicit recording/replay. No example API output has been invented for this repository.

In [ ]:
# Define the ticket and the yes/no question before making a request.
from typesafe_sdk import TypeSafeClient, Noul, RetryPolicy

state = {'ticket': 'I was charged twice. Please refund the duplicate.'}
questions = {
    'refund_requested': Noul(instructions='Does the customer explicitly request money back?'),
}
print('State analysed:')
display(state)

In [ ]:
# Send the evidence and question to Jev if live mode and a key are available.
first_response = None
if settings.mode != 'live' or not settings.typesafe_api_key:
    print('Set TYPESAFE_API_KEY and COURSE_MODE=live, then rerun setup.')
else:
    with TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    ) as client:
        first_response = client.system_one(
            model=settings.jev_new, state=state, questions=questions,
        )

In [ ]:
# Read the refund probability and the model that supplied it.
if first_response is None:
    print('No response yet. Run the request cell first.')
else:
    print('State analysed:')
    display(state)
    print('Refund-request probability:', first_response.nouls['refund_requested'].noul)
    print('Responding model:', first_response.model)
    display(first_response.model_dump(mode='json'))

## Choose the right mechanism

| Need | Starting point | Why |
|---|---|---|
| Is a refund requested? | Decision model | A bounded judgment about wording |
| Was the customer charged twice? | Database + code | Check authoritative transactions |
| Draft a considerate reply | Generative LLM | Requires original text |
| Compute refund total | Ordinary code | Arithmetic should be exact |

“System 1” and “System 2” are a useful analogy for fast judgment and deliberation, not a description of neurons or proof of architecture. TypeSafe describes Jev's training as reinforcement learning for calibrated decisions; training descriptions and vendor benchmark claims are not measurements of our workflow. We will test the tasks we care about.

## Exercise

Change the input to “I do not want a refund; please fix the invoice.” Predict the refund flag, run the call again, and inspect the number. Expected semantic answer: no. A model disagreement is useful evidence, not a reason to change the reference label automatically.

## Solution

Use the same Noul question and the new ticket. It should distinguish a mention of refunds from requesting one. Do not implement this as `'refund' in ticket`: that misses the negation.

## If setup fails

- Wrong import/kernel: select **Python (Jev course)** and restart.
- Missing/invalid key: edit `.env`; rerun setup. Never print the key.
- HTTP 429: reduce request frequency; this client allows one retry.
- Unavailable model or experimental feature: inspect the provider error and account access; do not silently switch models.
- Connection/timeout: check network access. Retry-inclusive latency is not pure model inference time.

Sources: [SDK](https://docs.typesafe.ai/sdk/python), [System One](https://docs.typesafe.ai/concepts/system-one), [models](https://docs.typesafe.ai/models).